# Chess Neural Network — Training Pipeline

Auto-detects training phase based on available checkpoints.
Runs on Kaggle T4/P100 GPU.

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/amiohossain/Chess.git /kaggle/working/Chess

# P100 (sm_60) requires PyTorch < 2.6
!pip install -q torch==2.5.1 torchvision==0.20.1 --force-reinstall --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r /kaggle/working/Chess/requirements.txt
!pip install -q zstandard

In [ ]:
import os, sys, glob, subprocess, json, shutil, time
sys.path.insert(0, '/kaggle/working/Chess')
os.makedirs('/kaggle/working/data', exist_ok=True)

HDF5_PATH = '/kaggle/working/supervised_positions.h5'

print("=" * 60)
print("DATA PREPARATION")
print("=" * 60)

# Check dataset input directory first
INPUT_DIR = '/kaggle/input/chess-training-data-2013'
if os.path.exists(INPUT_DIR):
    print(f'Dataset found at {INPUT_DIR}')
    !ls -la {INPUT_DIR}/
    h5_input = glob.glob(f'{INPUT_DIR}/**/*.h5', recursive=True)
    zst_input = sorted(glob.glob(f'{INPUT_DIR}/**/*.pgn.zst', recursive=True))
else:
    print(f'Dataset not mounted at {INPUT_DIR}. Downloading via API...')
    !kaggle datasets download amiohossain/chess-training-data-2013 --unzip -p /kaggle/working/chess-data
    INPUT_DIR = '/kaggle/working/chess-data'
    h5_input = glob.glob(f'{INPUT_DIR}/**/*.h5', recursive=True)
    zst_input = sorted(glob.glob(f'{INPUT_DIR}/**/*.pgn.zst', recursive=True))

# Use HDF5 if exists
if h5_input:
    print(f'Using existing HDF5: {h5_input[0]}')
    shutil.copy(h5_input[0], HDF5_PATH)
elif zst_input:
    total_zst = len(zst_input)
    total_zst_size = sum(os.path.getsize(z) for z in zst_input) / 1e9
    print(f'Found {total_zst} .pgn.zst files ({total_zst_size:.1f} GB total).')
    print(f'Decompressing and concatenating into a single PGN (max 200K positions)...')
    combined_pgn = '/kaggle/working/combined.pgn'

    for i, zst in enumerate(zst_input):
        zst_size = os.path.getsize(zst) / 1e6
        pgn_part = f'/kaggle/working/temp_{i}.pgn'
        print(f'[{i+1}/{total_zst}] {os.path.basename(zst)} ({zst_size:.0f} MB) -> decompressing...', end=' ')
        t0 = time.time()
        try:
            subprocess.run(['zstd', '-d', '-f', zst, '-o', pgn_part], check=True, capture_output=True)
        except:
            import zstandard as zstd_lib
            with open(zst, 'rb') as src, open(pgn_part, 'wb') as dst:
                zstd_lib.ZstdDecompressor().copy_stream(src, dst)
        pgn_size = os.path.getsize(pgn_part) / 1e6
        dt = time.time() - t0
        print(f'{pgn_size:.0f} MB ({dt:.1f}s)')

    # Concatenate all decompressed PGN parts
    print('Concatenating PGN parts...')
    t0 = time.time()
    with open(combined_pgn, 'wb') as dst:
        for i in range(total_zst):
            pgn_part = f'/kaggle/working/temp_{i}.pgn'
            with open(pgn_part, 'rb') as src:
                dst.write(src.read())
            os.remove(pgn_part)
    concat_time = time.time() - t0
    combined_mb = os.path.getsize(combined_pgn) / 1e6
    print(f'Combined PGN: {combined_mb:.0f} MB ({concat_time:.1f}s)')

    from src.data.pgn_processor import process_pgn
    n = process_pgn(pgn_path=combined_pgn, output_path=HDF5_PATH, max_positions=200_000)
    os.remove(combined_pgn)
    print(f'Result: {n} positions extracted to HDF5')
else:
    print('No data files found at all!')
    !ls -la {INPUT_DIR}/

if os.path.exists(HDF5_PATH):
    import h5py
    with h5py.File(HDF5_PATH, 'r') as f:
        n = f.attrs['num_positions']
    print(f'\nReady: {n} positions ready for training at {HDF5_PATH}')
else:
    raise FileNotFoundError(f'{HDF5_PATH} not found')

In [ ]:
# Run the training pipeline (auto-detects phase)
print("=" * 60)
print("TRAINING PIPELINE")
print("=" * 60)

import time as _time
_t0 = _time.time()
from src.kaggle.kaggle_main import run_session
run_session()
_t1 = _time.time()
print(f"\nTotal session time: {(_t1-_t0)/60:.1f} minutes")